# SARIMA Solar Irradiance Forecast

Seasonal ARIMA (SARIMA) time series forecasting pipeline for GHI (Global Horizontal Irradiance) with explicit annual seasonality handling, recursive 2025 predictions across multiple cities, and professional visualization with confidence intervals.

## 1. Imports and Configuration

This cell establishes the computational environment by loading all necessary libraries including pandas for data manipulation, numpy for numerical operations, matplotlib/seaborn for professional visualizations, and statsmodels for SARIMA modeling. The pmdarima auto_arima function enables automated parameter selection for optimal SARIMA (p,d,q)(P,D,Q,s) configurations. Global parameters are configured including file paths pointing to the parquet data source, target column specification (GHI), forecast horizon parameters (full year 2025), and visualization styling. Random seed settings ensure reproducibility across all forecasting runs.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from pmdarima import auto_arima
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error
from datetime import datetime
import sys
from joblib import Parallel, delayed
import multiprocessing

# Set random seed for reproducibility
np.random.seed(42)

# Configuration Parameters
PARQUET_PATH = '/workspaces/CSE-6242-Project/notebooks/irradiance_2021_2024.parquet'
TARGET_COLUMN = 'GHI'
FORECAST_START = pd.Timestamp('2025-01-01 00:00:00')
FORECAST_END = pd.Timestamp('2025-12-31 23:00:00')
OUTPUT_PARQUET = 'sarima_forecasts_by_city.parquet'

# SARIMA Configuration
SEASONAL_PERIOD = 24 * 365  # Annual seasonality for hourly data
AUTO_ARIMA_SEASONAL_PERIOD = 365  # Use daily aggregates for auto_arima (faster)
AUTO_ARIMA_MAX_P = 3
AUTO_ARIMA_MAX_Q = 3
AUTO_ARIMA_MAX_P_SEASONAL = 2
AUTO_ARIMA_MAX_Q_SEASONAL = 2
AUTO_ARIMA_MAX_D = 2
AUTO_ARIMA_MAX_D_SEASONAL = 1

# Parallel Processing Configuration
NUM_JOBS = -1  # Use all available CPU cores (-1 means all cores)

# Set professional plot styling
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': '#f8f9fa',
    'grid.color': '#e0e0e0',
    'grid.linestyle': ':',
    'grid.linewidth': 0.5,
})

print("✓ All imports and configurations complete")
print(f"✓ Parallel processing enabled: {multiprocessing.cpu_count()} CPU cores available")

✓ All imports and configurations complete
✓ Parallel processing enabled: 2 CPU cores available


## 2. Data Cleaning and Feature Engineering

This section loads hourly GHI measurements from the parquet file covering 2021-2024 and performs comprehensive data preparation. The exploratory analysis examines dataset structure, identifies all available geographic locations (cities), and computes summary statistics. Data quality checks include counting missing values, verifying temporal continuity, and examining GHI distributions. The city column is auto-detected to accommodate different dataset formats. For SARIMA modeling, data is aggregated to daily averages to reduce computational complexity while preserving essential seasonal patterns. Daily aggregation transforms 8,760 hourly records per year into 365 daily records, enabling efficient parameter selection and model training. Summary statistics display mean, standard deviation, minimum, and maximum values to establish baseline understanding of solar irradiance patterns before modeling.

In [2]:
print("="*70)
print("DATA LOADING AND PREPARATION")
print("="*70)

# Load parquet file
df = pd.read_parquet(PARQUET_PATH)
ghi_series = df[TARGET_COLUMN].copy()

print(f"\n📊 DATASET OVERVIEW")
print(f"  Shape: {df.shape}")
print(f"  Date range: {df.index.min()} to {df.index.max()}")
print(f"  Total hours: {len(df):,}")
print(f"  Memory usage: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")

print(f"\n📈 {TARGET_COLUMN} STATISTICS (HOURLY)")
print(f"  Mean: {ghi_series.mean():.2f} W/m²")
print(f"  Std Dev: {ghi_series.std():.2f} W/m²")
print(f"  Min: {ghi_series.min():.2f} W/m²")
print(f"  Max: {ghi_series.max():.2f} W/m²")
print(f"  Missing values: {ghi_series.isnull().sum():,}")
print(f"  Percentiles (25/50/75): {ghi_series.quantile(0.25):.1f} / {ghi_series.quantile(0.5):.1f} / {ghi_series.quantile(0.75):.1f}")

# Detect city column
city_column = None
if 'city' in df.columns:
    city_column = 'city'
elif 'City' in df.columns:
    city_column = 'City'

if city_column is not None:
    unique_cities = sorted(df[city_column].unique())
    print(f"\n🌍 CITIES DETECTED ({city_column})")
    print(f"  Total unique cities: {len(unique_cities)}")
    for i, city in enumerate(unique_cities, 1):
        city_count = (df[city_column] == city).sum()
        print(f"  {i}. {city}: {city_count:,} records")
else:
    print(f"\n⚠ Warning: City column not found - will process as single dataset")
    unique_cities = [None]

# Aggregate to daily averages for SARIMA modeling
print(f"\n📅 AGGREGATING TO DAILY AVERAGES")
daily_ghi = df.resample('D')[TARGET_COLUMN].mean()
print(f"  Original hourly records: {len(df):,}")
print(f"  Daily aggregated records: {len(daily_ghi):,}")
print(f"  Daily GHI statistics:")
print(f"    Mean: {daily_ghi.mean():.2f} W/m²")
print(f"    Std Dev: {daily_ghi.std():.2f} W/m²")
print(f"    Min: {daily_ghi.min():.2f} W/m²")
print(f"    Max: {daily_ghi.max():.2f} W/m²")

print("\n✓ Data loading and preparation complete")

DATA LOADING AND PREPARATION

📊 DATASET OVERVIEW
  Shape: (245280, 11)
  Date range: 2021-01-01 00:30:00 to 2024-12-31 23:30:00
  Total hours: 245,280
  Memory usage: 27.5 MB

📈 GHI STATISTICS (HOURLY)
  Mean: 192.15 W/m²
  Std Dev: 281.20 W/m²
  Min: 0.00 W/m²
  Max: 1098.00 W/m²
  Missing values: 0
  Percentiles (25/50/75): 0.0 / 6.0 / 334.0

🌍 CITIES DETECTED (City)
  Total unique cities: 7
  1. Atlanta: 35,040 records
  2. Boston: 35,040 records
  3. Chicago: 35,040 records
  4. Denver: 35,040 records
  5. Los Angeles: 35,040 records
  6. Phoenix: 35,040 records
  7. Seattle: 35,040 records

📅 AGGREGATING TO DAILY AVERAGES
  Original hourly records: 245,280
  Daily aggregated records: 1,461
  Daily GHI statistics:
    Mean: 192.15 W/m²
    Std Dev: 76.57 W/m²
    Min: 42.57 W/m²
    Max: 342.92 W/m²

✓ Data loading and preparation complete


## 3. Model Training and Validation

This section trains SARIMA models for each city using automated parameter selection followed by model fitting. For each city, the pmdarima auto_arima function identifies optimal (p,d,q)(P,D,Q,s) parameters by evaluating multiple configurations and selecting the combination with lowest AIC (Akaike Information Criterion). The seasonal period s is set to 365 (daily aggregates), capturing annual solar irradiance cycles. Search bounds constrain p, d, q, P, D, Q to reasonable ranges to balance model complexity with computational efficiency. Once optimal parameters are identified, a full SARIMAX model is trained on the complete historical daily time series. Training metrics (MAE, RMSE, MAPE on held-out test data) quantify model accuracy. A simple progress indicator tracks completion across cities. All fitted models and training metrics are stored per-city for use in recursive forecasting.

In [3]:
print("\n" + "="*70)
print("MODEL TRAINING AND VALIDATION")
print("="*70)

def train_city_model(city, df, city_column, target_column):
    """
    Train SARIMA model for a single city.
    
    Returns a tuple: (city_name, model_dict, metrics_dict) or (city_name, None, None) if training failed
    """
    try:
        # Filter data for this city
        if city_column is not None and city is not None:
            city_df = df[df[city_column] == city].copy()
        else:
            city_df = df.copy()
        
        city_df = city_df.sort_index()
        city_daily_ghi = city_df.resample('D')[target_column].mean()
        
        city_key = city if city else 'Unknown'
        
        # Check data sufficiency
        if len(city_daily_ghi) < 730:
            return (city_key, None, None)
        
        try:
            # Auto ARIMA with seasonal component
            auto_model = auto_arima(
                city_daily_ghi,
                seasonal=True,
                m=AUTO_ARIMA_SEASONAL_PERIOD,
                max_p=AUTO_ARIMA_MAX_P,
                max_q=AUTO_ARIMA_MAX_Q,
                max_d=AUTO_ARIMA_MAX_D,
                max_P=AUTO_ARIMA_MAX_P_SEASONAL,
                max_Q=AUTO_ARIMA_MAX_Q_SEASONAL,
                max_D=AUTO_ARIMA_MAX_D_SEASONAL,
                stepwise=True,
                trace=False,
                error_action='ignore',
                suppress_warnings=True,
                information_criterion='aic',
                random_state=42
            )
            
            optimal_order = auto_model.order
            optimal_seasonal_order = auto_model.seasonal_order
            
        except Exception as e:
            optimal_order = (1, 1, 1)
            optimal_seasonal_order = (1, 1, 1, 365)
        
        # Train full SARIMAX model
        model = SARIMAX(
            city_daily_ghi,
            order=optimal_order,
            seasonal_order=optimal_seasonal_order,
            enforce_stationarity=False,
            enforce_invertibility=False
        )
        
        model_fit = model.fit(disp=False, maxiter=500)
        
        # Get training metrics using last year of data as test
        test_size = 365
        if len(city_daily_ghi) > test_size:
            y_train = city_daily_ghi[:-test_size]
            y_test = city_daily_ghi[-test_size:]
            
            # Make predictions on test set
            predictions = model_fit.get_forecast(steps=test_size).predicted_mean
            
            train_mae = mean_absolute_error(y_test, predictions)
            train_rmse = np.sqrt(mean_squared_error(y_test, predictions))
            
            # MAPE only on non-zero periods
            mask_nonzero = y_test.values > 5
            if mask_nonzero.sum() > 0:
                train_mape = mean_absolute_percentage_error(
                    y_test.values[mask_nonzero],
                    predictions.values[mask_nonzero] + 1e-10
                ) * 100
            else:
                train_mape = 0.0
        else:
            train_mae = 0.0
            train_rmse = 0.0
            train_mape = 0.0
        
        # Return results
        model_dict = {
            'model_fit': model_fit,
            'order': optimal_order,
            'seasonal_order': optimal_seasonal_order,
            'metrics': {'mae': train_mae, 'rmse': train_rmse, 'mape': train_mape}
        }
        
        metrics_dict = {
            'city': city_key,
            'mae': train_mae,
            'rmse': train_rmse,
            'mape': train_mape,
            'training_days': len(y_train) if len(city_daily_ghi) > test_size else len(city_daily_ghi)
        }
        
        return (city_key, model_dict, metrics_dict)
        
    except Exception as e:
        city_key = city if city else 'Unknown'
        return (city_key, None, None)

# Train models in parallel
print(f"\n🚀 PARALLEL TRAINING: {len(unique_cities)} city/cities on {multiprocessing.cpu_count()} cores")
print("  (Training up to {} cities simultaneously...)\n".format(min(len(unique_cities), multiprocessing.cpu_count())))

results = Parallel(n_jobs=NUM_JOBS, verbose=10)(
    delayed(train_city_model)(city, df, city_column, TARGET_COLUMN)
    for city in unique_cities
)

# Process results
trained_models = {}
city_metrics = []

for city_key, model_dict, metrics_dict in results:
    if model_dict is not None:
        trained_models[city_key] = model_dict
        city_metrics.append(metrics_dict)
        print(f"\n✓ {city_key}: MAE={metrics_dict['mae']:.2f} W/m² | RMSE={metrics_dict['rmse']:.2f} W/m² | MAPE={metrics_dict['mape']:.2f}%")
    else:
        print(f"\n✗ {city_key}: Training failed or insufficient data")

print(f"\n{'='*70}")
print(f"✓ Parallel model training complete for {len(trained_models)} cities")
print(f"  Successfully trained: {len(trained_models)}/{len(unique_cities)} cities")


MODEL TRAINING AND VALIDATION

🚀 PARALLEL TRAINING: 7 city/cities on 2 cores
  (Training up to 2 cities simultaneously...)



[Parallel(n_jobs=-1)]: Using backend LokyBackend with 2 concurrent workers.
/usr/local/python/3.12.1/lib/python3.12/site-packages/statsmodels/tsa/statespace/sarimax.py:866: UserWarning: Too few observations to estimate starting parameters for seasonal ARMA. All parameters except for variances will be set to zeros.
  warn('Too few observations to estimate starting parameters%s.'
/usr/local/python/3.12.1/lib/python3.12/site-packages/statsmodels/tsa/statespace/sarimax.py:866: UserWarning: Too few observations to estimate starting parameters for seasonal ARMA. All parameters except for variances will be set to zeros.
  warn('Too few observations to estimate starting parameters%s.'


KeyboardInterrupt: 

## 4. Forecasting Full Year 2025

This section generates recursive hourly forecasts for the entire 2025 year (8,760 total hours) for each trained city model. For each city, daily forecasts are generated first using the fitted SARIMAX model over the 365-day forecast horizon. Daily predictions are then expanded to hourly frequency by replicating daily values across 24 hours (representing average hourly values throughout each day). This approach maintains consistency with training data while producing operational hourly output. Each forecast record includes associated model metrics and timestamps. A simple progress indicator tracks forecast generation across cities and dates. All forecast records are appended to a results list for subsequent visualization and export.

In [ ]:
print("\n" + "="*70)
print("GENERATING FULL YEAR 2025 FORECASTS")
print("="*70)

all_results = []
future_dates_daily = pd.date_range(start=FORECAST_START, end=FORECAST_END, freq='D')

for city_idx, (city_name, model_dict) in enumerate(trained_models.items(), 1):
    print(f"\n[{city_idx}/{len(trained_models)}] 🔮 Generating forecasts for {city_name}...")
    
    model_fit = model_dict['model_fit']
    metrics = model_dict['metrics']
    
    try:
        # Generate daily forecasts for 365 days of 2025
        forecast_result = model_fit.get_forecast(steps=365)
        daily_forecasts = forecast_result.predicted_mean
        
        print(f"  Generated {len(daily_forecasts)} daily forecasts")
        
        # Expand daily forecasts to hourly
        hourly_count = 0
        for day_idx, (date, daily_value) in enumerate(daily_forecasts.items()):
            # Progress indicator every 50 days
            if day_idx % 50 == 0 and day_idx > 0:
                print(f"    → {day_idx}/365 days processed")
            
            # Ensure non-negative forecast
            daily_value = max(0.0, daily_value)
            
            # Create hourly entries for this day
            for hour in range(24):
                hourly_date = date + pd.Timedelta(hours=hour)
                if hourly_date >= FORECAST_START and hourly_date <= FORECAST_END:
                    all_results.append({
                        'city': city_name,
                        'date': hourly_date,
                        'actual_ghi': np.nan,
                        'forecasted_ghi': daily_value,  # Same value across 24 hours
                        'mae': metrics['mae'],
                        'rmse': metrics['rmse'],
                        'mape': metrics['mape'],
                        'data_type': 'forecast'
                    })
                    hourly_count += 1
        
        print(f"  ✓ Generated {hourly_count:,} hourly forecast records")
        
    except Exception as e:
        print(f"  ✗ Forecasting failed: {e}")
        continue

print(f"\n{'='*70}")
print(f"✓ Forecast generation complete!")
print(f"✓ Total forecast records generated: {len(all_results):,}")

## 5. Historical vs Forecast Visualizations

This section creates publication-quality individual forecast charts for each city displaying the complete data timeline. Historical GHI data (2021-2024) is rendered in black, while 2025 daily aggregate forecasts appear in bright blue with 95% confidence interval bands (1.96 × RMSE) shown as faint blue fill regions. A vertical dashed line clearly demarcates the transition point between historical and forecast periods. Professional formatting includes clean styling, optimized color palettes, high-resolution output (300 DPI PNG), and comprehensive annotations. Each plot includes model performance statistics (MAE, RMSE, MAPE) in a rounded statistics box, legend with interpretable descriptions, and properly formatted axes. Grid styling uses faint dashed lines for understandability. The visualization enables visual assessment of forecast quality, seasonal amplitude capture, and confidence bounds for each geographic location.

In [ ]:
print("\n" + "="*70)
print("CREATING PROFESSIONAL HISTORICAL VS FORECAST VISUALIZATIONS")
print("="*70)

# Add historical data to results
for city in unique_cities:
    if city_column is not None and city is not None:
        city_df = df[df[city_column] == city].copy()
    else:
        city_df = df.copy()
    
    city_df = city_df.sort_index()
    
    # Get metrics from trained model
    city_key = city if city else 'Unknown'
    if city_key in trained_models:
        metrics = trained_models[city_key]['metrics']
    else:
        metrics = {'mae': 0, 'rmse': 0, 'mape': 0}
    
    for ts, ghi_value in city_df[TARGET_COLUMN].items():
        all_results.append({
            'city': city_key,
            'date': ts,
            'actual_ghi': ghi_value,
            'forecasted_ghi': np.nan,
            'mae': metrics['mae'],
            'rmse': metrics['rmse'],
            'mape': metrics['mape'],
            'data_type': 'historical'
        })

# Convert results to DataFrame
results_df = pd.DataFrame(all_results)

# Get unique cities
cities_list = sorted(results_df['city'].unique())

for city in cities_list:
    city_data = results_df[results_df['city'] == city].copy()
    city_data = city_data.sort_values('date')
    
    # Separate historical and forecasted data
    historical_data = city_data[city_data['actual_ghi'].notna()].copy()
    forecast_data = city_data[city_data['actual_ghi'].isna()].copy()
    
    if len(historical_data) == 0:
        print(f"⚠ No historical data for {city}, skipping...")
        continue
    
    # Calculate confidence interval
    rmse = city_data['rmse'].iloc[0]
    confidence_interval = 1.96 * rmse
    
    # Create figure
    fig, ax = plt.subplots(figsize=(14, 7))
    
    # Plot historical data in black
    if len(historical_data) > 0:
        ax.plot(
            historical_data['date'],
            historical_data['actual_ghi'],
            color='#1f1f1f',
            linewidth=2.2,
            label='Historical Data (2021-2024)',
            zorder=3,
            alpha=0.95
        )
    
    # Plot forecast with confidence interval
    if len(forecast_data) > 0:
        forecast_data = forecast_data.copy()
        forecast_data['upper_ci'] = (forecast_data['forecasted_ghi'] + confidence_interval).clip(lower=0)
        forecast_data['lower_ci'] = (forecast_data['forecasted_ghi'] - confidence_interval).clip(lower=0)
        
        # Confidence interval band
        ax.fill_between(
            forecast_data['date'],
            forecast_data['lower_ci'],
            forecast_data['upper_ci'],
            color='#4a90e2',
            alpha=0.15,
            label=f'95% Confidence Interval (±{confidence_interval:.1f} W/m²)',
            zorder=1
        )
        
        # Forecast line
        ax.plot(
            forecast_data['date'],
            forecast_data['forecasted_ghi'],
            color='#2563eb',
            linewidth=2.2,
            label='GHI Forecast (2025)',
            zorder=2,
            alpha=0.9
        )
    
    # Vertical separator line
    if len(historical_data) > 0 and len(forecast_data) > 0:
        transition_point = historical_data['date'].iloc[-1]
        ax.axvline(
            x=transition_point,
            color='#666666',
            linestyle='--',
            linewidth=1.5,
            alpha=0.5,
            zorder=1.5,
            label='Forecast Start'
        )
    
    # Formatting
    ax.set_xlabel('Time', fontsize=12, fontweight='600', color='#333333')
    ax.set_ylabel('Global Horizontal Irradiance (GHI) [W/m²]', fontsize=12, fontweight='600', color='#333333')
    ax.set_title(f'Solar Irradiance Forecast - {city}\nHistorical Data vs 2025 SARIMA Forecast',
                fontsize=14, fontweight='700', color='#1a1a1a', pad=20)
    
    ax.grid(True, alpha=0.3, linestyle=':', linewidth=0.7)
    ax.set_axisbelow(True)
    
    # Legend
    ax.legend(loc='upper left', frameon=True, fancybox=False, shadow=False,
             framealpha=0.95, edgecolor='#cccccc', fontsize=10)
    
    # Styling
    ax.tick_params(colors='#333333', which='both', length=5)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{int(x)}'))
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_color('#cccccc')
    ax.spines['bottom'].set_color('#cccccc')
    
    # Statistics box
    stats_text = (
        f"Model Performance:\n"
        f"MAE:  {city_data['mae'].iloc[0]:.2f} W/m²\n"
        f"RMSE: {city_data['rmse'].iloc[0]:.2f} W/m²\n"
        f"MAPE: {city_data['mape'].iloc[0]:.2f}%"
    )
    
    ax.text(0.98, 0.02, stats_text, transform=ax.transAxes,
           fontsize=9, verticalalignment='bottom', horizontalalignment='right',
           bbox=dict(boxstyle='round', facecolor='white', alpha=0.8, edgecolor='#cccccc', linewidth=1),
           family='monospace')
    
    plt.tight_layout()
    
    # Save figure
    filename = f'sarima_ghi_forecast_{city.lower().replace(" ", "_")}.png'
    plt.savefig(filename, dpi=300, bbox_inches='tight', facecolor='white')
    print(f"✓ Saved: {filename}")
    
    plt.show()

print(f"\n✓ Professional visualizations complete!")

## 6. Export Results to Parquet

This final section persists the complete forecast dataset (historical + 2025 predictions) to columnar parquet format for downstream analysis and consumption by external pipelines. The export preserves all metadata including city identifier, date timestamp, actual_ghi (historical only), forecasted_ghi (forecast only), and model metrics (mae, rmse, mape). Parquet format provides efficient compression, schema validation, and fast read performance for analytical queries. Data integrity checks verify row count matches expectations and confirm all required columns are present. The file persists to 'sarima_forecasts_by_city.parquet' in the current working directory, enabling reproducible end-to-end workflow execution and facilitating data sharing with downstream analysis teams.

In [ ]:
print("\n" + "="*70)
print("EXPORTING RESULTS TO PARQUET")
print("="*70)

# Create export DataFrame
export_df = results_df.copy()
export_df = export_df.set_index('date')

print(f"\n📋 EXPORT DATAFRAME")
print(f"  Shape: {export_df.shape}")
print(f"  Columns: {list(export_df.columns)}")
print(f"  Index: {export_df.index.name}")
print(f"  Date range: {export_df.index.min()} to {export_df.index.max()}")

# Verify data completeness
print(f"\n✓ DATA VALIDATION")
print(f"  Total rows: {len(export_df):,}")
print(f"  Cities: {export_df['city'].nunique()}")
print(f"  Date range span: {(export_df.index.max() - export_df.index.min()).days + 1} days")
print(f"  Missing 'city': {export_df['city'].isna().sum()}")
print(f"  Missing 'mae': {export_df['mae'].isna().sum()}")

# Historical data checks
historical_df = export_df[export_df['actual_ghi'].notna()]
print(f"\n  Historical records: {len(historical_df):,}")
if len(historical_df) > 0:
    print(f"    Date range: {historical_df.index.min()} to {historical_df.index.max()}")
    print(f"    GHI range: {historical_df['actual_ghi'].min():.1f} - {historical_df['actual_ghi'].max():.1f} W/m²")

# Forecast data checks
forecast_df = export_df[export_df['forecasted_ghi'].notna()]
print(f"\n  Forecast records: {len(forecast_df):,}")
if len(forecast_df) > 0:
    print(f"    Date range: {forecast_df.index.min()} to {forecast_df.index.max()}")
    print(f"    GHI range: {forecast_df['forecasted_ghi'].min():.1f} - {forecast_df['forecasted_ghi'].max():.1f} W/m²")

# Export to parquet
print(f"\n💾 WRITING TO PARQUET...")
try:
    export_df.to_parquet(OUTPUT_PARQUET, compression='snappy', index=True)
    print(f"✓ Successfully saved: {OUTPUT_PARQUET}")
    
    # Verify file was written
    import os
    file_size = os.path.getsize(OUTPUT_PARQUET) / 1e6
    print(f"  File size: {file_size:.1f} MB")
    
    # Verify by reading back
    print(f"\n✓ VERIFICATION")
    verify_df = pd.read_parquet(OUTPUT_PARQUET)
    print(f"  Read back shape: {verify_df.shape}")
    print(f"  Columns match: {list(verify_df.columns) == list(export_df.columns)}")
    print(f"  Index name: {verify_df.index.name}")
    
    # Display sample records
    print(f"\n📋 SAMPLE RECORDS (first 5 forecasts)")
    sample_df = verify_df[verify_df['forecasted_ghi'].notna()].head()
    print(sample_df.to_string())
    
except Exception as e:
    print(f"✗ ERROR writing parquet: {e}")

print(f"\n{'='*70}")
print(f"✓ WORKFLOW COMPLETE")
print(f"{'='*70}")
print(f"\n📊 SUMMARY")
print(f"  Historical records: {len(historical_df):,}")
print(f"  Forecast records: {len(forecast_df):,}")
print(f"  Total records: {len(export_df):,}")
print(f"  Cities processed: {export_df['city'].nunique()}")
print(f"  Output file: {OUTPUT_PARQUET}")
print(f"\n✓ All SARIMA forecasts saved and ready for analysis!")